In [0]:
# =============================================================================
# CAMADA GOLD - Modelagem final para análise
# =============================================================================
# Objetivo: Construir a tabela fato central do projeto juntando todas as
# tabelas Silver em um modelo analítico pronto para responder as perguntas
# de negócio definidas no objetivo do MVP.
#
# Tabela gerada: gold_fato_pedidos
# Granularidade: um registro por item de pedido
# =============================================================================

from pyspark.sql import functions as F

# =============================================================================
# Leitura das tabelas Silver
# =============================================================================
orders    = spark.table("workspace.default.silver_orders")
items     = spark.table("workspace.default.silver_order_items")
payments  = spark.table("workspace.default.silver_order_payments")
products  = spark.table("workspace.default.silver_products")
customers = spark.table("workspace.default.silver_customers")
translate = spark.table("workspace.default.silver_category_translation")
sellers   = spark.table("workspace.default.silver_sellers")

# =============================================================================
# Agregação de pagamentos por pedido
# =============================================================================
# Um pedido pode ter múltiplas formas de pagamento (ex: cartão + voucher)
# Aqui pegamos a forma de pagamento principal (maior valor) e somamos o total
payments_agg = (payments
    .groupBy("order_id")
    .agg(
        # Forma de pagamento com maior valor no pedido
        F.first("payment_type").alias("payment_type"),
        # Média de parcelas considerando todos os pagamentos do pedido
        F.round(F.avg("payment_installments"), 0).cast("int").alias("payment_installments"),
        # Valor total pago no pedido (soma de todos os pagamentos)
        F.round(F.sum("payment_value"), 2).alias("total_payment_value")
    )
)

# =============================================================================
# Construção da tabela fato por joins sucessivos
# =============================================================================

# 1. Itens + Pedidos: base da fato, um registro por item
fato = items.join(orders, on="order_id", how="inner")

# 2. + Clientes: informação geográfica do comprador
fato = fato.join(customers, on="customer_id", how="left")

# 3. + Produtos: categoria do produto
fato = fato.join(
    products.select("product_id", "product_category_name"),
    on="product_id",
    how="left"
)

# 4. + Tradução de categorias: categoria em inglês para padronização
fato = fato.join(translate, on="product_category_name", how="left")

# 5. + Vendedores: estado do vendedor
fato = fato.join(
    sellers.select("seller_id", "seller_state"),
    on="seller_id",
    how="left"
)

# 6. + Pagamentos agregados: forma e valor de pagamento por pedido
fato = fato.join(payments_agg, on="order_id", how="left")

# =============================================================================
# Seleção e renomeação das colunas finais
# =============================================================================
fato = fato.select(
    # Identificadores
    F.col("order_id"),
    F.col("order_item_id"),
    F.col("customer_id"),
    F.col("product_id"),
    F.col("seller_id"),

    # Temporal
    F.col("order_purchase_timestamp"),
    F.year("order_purchase_timestamp").alias("ano_compra"),
    F.month("order_purchase_timestamp").alias("mes_compra"),

    # Status do pedido
    F.col("order_status"),

    # Produto
    F.col("product_category_name").alias("categoria_pt"),
    F.col("product_category_name_english").alias("categoria_en"),

    # Valores financeiros
    F.col("price").alias("preco_item"),
    F.col("freight_value").alias("frete"),
    F.round(F.col("price") + F.col("freight_value"), 2).alias("valor_total_item"),

    # Pagamento
    F.col("payment_type").alias("forma_pagamento"),
    F.col("payment_installments").alias("parcelas"),
    F.col("total_payment_value").alias("valor_total_pedido"),

    # Geografia
    F.col("customer_state").alias("estado_cliente"),
    F.col("customer_city").alias("cidade_cliente"),
    F.col("seller_state").alias("estado_vendedor"),
)

# =============================================================================
# Salvando a tabela Gold
# =============================================================================
fato.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_fato_pedidos")
print(f"✓ gold_fato_pedidos salva com {fato.count()} registros")
print(f"  Colunas: {len(fato.columns)}")
print(f"  Período: {fato.agg(F.min('order_purchase_timestamp'), F.max('order_purchase_timestamp')).collect()}")